In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import xarray as xr
import multiprocessing as mp
from matplotlib.patches import Patch
import numpy as np
import rioxarray

# Check Global Land Cover Map

In [ ]:
#CCI LC data 
ds = xr.open_dataset("/net/krypton/hyclimm/lixinh/CCI_LC_maps/nc_world/ESACCI-LC-L4-LCCS-Map-300m-P1Y-2001-v2.0.7cds.nc")

In [ ]:
ds   #coordination system not properly assigned, but it is in WGS84 (weird point!)
ds = ds.rio.write_crs("EPSG:4326")

In [ ]:
ds.rio.crs

In [ ]:
#use the MODIS regionalization to clip out Europe (according to FireCCI user guideline)
clipped_ds = ds.sel(lat=slice(83, 25), lon=slice(-26, 53))

In [ ]:
clipped_ds.rio.crs

# Clip to Europe–North Africa Region and Save

In [ ]:
#define clipping function
def Clip_CCI_LC(yr):
    
    if yr <= 2015:
        ds = xr.open_dataset(f"/net/krypton/hyclimm/lixinh/CCI_LC_maps/nc_world/ESACCI-LC-L4-LCCS-Map-300m-P1Y-{yr}-v2.0.7cds.nc")
    else:
        ds = xr.open_dataset(f"/net/krypton/hyclimm/lixinh/CCI_LC_maps/nc_world/C3S-LC-L4-LCCS-Map-300m-P1Y-{yr}-v2.1.1.nc")

    #assign coordination system WGS84
    ds = ds.rio.write_crs('EPSG:4326')

    #use the MODIS regionalization to clip out Europe (according to FireCCI user guideline)
    clipped_ds = ds.sel(lat=slice(83, 25), lon=slice(-26, 53))
    clipped_ds = clipped_ds.rio.write_crs('EPSG:4326')

    #save
    if yr <= 2015:
        clipped_ds.to_netcdf(f"/net/krypton/hyclimm/data/projects/SynFire/WP1/CCI_Land_Cover_Maps_EU/EU_ESACCI-LC-L4-LCCS-Map-300m-P1Y-{yr}-v2.0.7cds.nc")
    else:
        clipped_ds.to_netcdf(f"/net/krypton/hyclimm/data/projects/SynFire/WP1/CCI_Land_Cover_Maps_EU/EU_C3S-LC-L4-LCCS-Map-300m-P1Y-{yr}-v2.1.1.nc")

In [ ]:
#map over 2000-2022
mp.cpu_count() #64
pool = mp.Pool(5)    #specify number of cores
pool.map(Clip_CCI_LC, [year for year in range(2000, 2023)])
pool.close()

# Visualisation

In [ ]:
ds = xr.open_dataset("/net/krypton/hyclimm/data/projects/SynFire/WP1/CCI_Land_Cover_Maps_EU/EU_ESACCI-LC-L4-LCCS-Map-300m-P1Y-2001-v2.0.7cds.nc")

In [ ]:
print(np.unique(ds["lccs_class"].values))

In [ ]:
ds

In [ ]:
ds.rio.crs #crs is still not there, need to be re-assigned when loading the data

In [ ]:
for yr in range(2000, 2023):
    
    # Define colormaps
    colors = ['black', '#d5b41b', '#448639', '#62a29e', 'red', '#d7c49b', 'blue', 'grey']
    labels = ['No Data', 'Cropland', 'Natural Vegetation', 'Suspicious Water', 'Urban', 'Bare Land', 'Water Bodies', 'Snow and Ice'] 
    cmap = ListedColormap(colors)
    
    bounds = [0, 5, 45, 155, 185, 195, 205, 215, 225] #len(bounds) = len(colors) + 1
    norm = BoundaryNorm(bounds, cmap.N)
    
    #Data
    if yr < 2016:
        ds = xr.open_dataset(f"/net/krypton/hyclimm/data/projects/SynFire/WP1/CCI_Land_Cover_Maps_EU/EU_ESACCI-LC-L4-LCCS-Map-300m-P1Y-{yr}-v2.0.7cds.nc")
    else:
        ds = xr.open_dataset(f"/net/krypton/hyclimm/data/projects/SynFire/WP1/CCI_Land_Cover_Maps_EU/EU_C3S-LC-L4-LCCS-Map-300m-P1Y-{yr}-v2.1.1.nc")
    
    LC_class = ds['lccs_class'].squeeze('time')
    
    #extent
    lon = ds.lon.values
    lat = ds.lat.values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]
    
    
    # Plot the data with the custom color scale
    plt.figure(figsize=(10, 6))
    img = plt.imshow(LC_class, cmap=cmap, norm=norm, extent=extent)
    plt.title(f'CCI Land Cover {yr}')
    
    # Create custom legend handles
    handles = [Patch(color=colors[i], label=labels[i]) for i in range(len(labels))]
    
    # Add the legend to the plot
    plt.legend(handles=handles, loc='upper right', bbox_to_anchor=(1.35, 1), title='Land Type')
    
    plt.show()